<a href="https://colab.research.google.com/github/arinjay-singh/econ3916-statistical-machine-learning/blob/main/Class%206%20/%20class6_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Ingestion and Manual Shuffling

In [15]:
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Data Ingestion (The Population)
df = sns.load_dataset('titanic')
print(f"Total Population: {len(df)}")
print(f"Population Survival Rate: {df['survived'].mean():.4f}")

# 2. Manual Shuffle (Simulation of Sampling)
# We set a seed to ensure reproducibility for the lesson,
# but in production, this variance happens naturally.
np.random.seed(2026)
indices = np.random.permutation(len(df))

Total Population: 891
Population Survival Rate: 0.3838


## Step 2: The Split and The Bias Check

In [16]:
# 3. Cut the deck (80/20 Split)
split_point = int(0.8 * len(df))

# Slicing the shuffled indices
train_idx = indices[:split_point]
test_idx = indices[split_point:]

# Creating the subsets
train_set = df.iloc[train_idx]
test_set = df.iloc[test_idx]

# 4. Bias Check (The Delta)
train_surv = train_set['survived'].mean()
test_surv = test_set['survived'].mean()
delta = abs(train_surv - test_surv)

print(f"Train Survival Rate: {train_surv:.4f}")
print(f"Test Survival Rate:  {test_surv:.4f}")
print(f"Sampling Bias (Delta): {delta:.4f}")

Train Survival Rate: 0.3736
Test Survival Rate:  0.4246
Sampling Bias (Delta): 0.0510


## Step 3: Fixing Covariate Shift

In [17]:
from sklearn.model_selection import train_test_split

# Stratify by 'pclass' ensures the distribution of classes is identical
X_train, X_test = train_test_split(df, test_size=0.2, stratify=df['pclass'])

print("\n--- Stratified Split ---")
print("Train Class Dist:\n", X_train['pclass'].value_counts(normalize=True))
print("Test Class Dist:\n", X_test['pclass'].value_counts(normalize=True))


--- Stratified Split ---
Train Class Dist:
 pclass
3    0.550562
1    0.242978
2    0.206461
Name: proportion, dtype: float64
Test Class Dist:
 pclass
3    0.553073
1    0.240223
2    0.206704
Name: proportion, dtype: float64


## Step 4: The SRM Diagnostic (Forensics)

In [18]:
from scipy.stats import chisquare

observed = [450, 550]
expected = [500, 500]

chi2, p_value = chisquare(observed, f_exp=expected)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.01:
    print("CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.")
else:
    print("Variance is within natural limits.")

Chi-Square Statistic: 10.0000
P-value: 0.001565
CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.


### Why 550/450 isn't just "bad luck":
A fair 50/50 randomizer on 1,000 users produces a binomial distribution centered at 500 with a standard deviation of ~15.8 (√(1000 × 0.5 × 0.5)). A 550/450 split is a 50-user deviation — over 3σ away from the mean. Under a fair coin, the probability of drifting this far or farther is ~0.16%, well below the 1% significance threshold.
This matters because SRM doesn't just mean "unlucky flip" — it signals a systematic upstream bug: a broken load balancer, bot traffic disproportionately hitting one variant, redirect failures, or a triggering condition that's correlated with the variant assignment. Any of these contaminate both groups non-randomly, meaning your treatment effect estimate is biased regardless of how large your sample gets. You should halt analysis and root-cause the allocation pipeline before interpreting any metric deltas.